# ReVA Video Analysis

Downloads only three documented small example videos from the pinned revision, measures their metadata and displays sampled frames. This is a sample analysis, not a dataset-wide estimate.


In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from preprocessing.hf_access import fetch_dataset_file
from preprocessing.video_loader import get_video_metadata, read_frames
from preprocessing.frame_sampling import uniform_indices
manifest=json.loads(Path("manifests/reva_manifest.json").read_text(encoding="utf-8"))
repo,revision=manifest["repo_id"],manifest["revision"]
sample_paths=["Hawk_UAV/NJ/DJI_0157_d4_01.mp4","VisDrone/uav0000009_03358_v_01.mp4","Hawk_UAV/BE/DJI_0092_d4_04.mp4"]


In [ ]:
records=[]; decoded={}; failures=[]
for rel in sample_paths:
    try:
        path=fetch_dataset_file(repo,rel,revision=revision)
        meta=get_video_metadata(path)
        ids=uniform_indices(meta["frame_count"],8).tolist()
        frames=read_frames(path,ids)
        records.append({"video_path":rel,"file_bytes":path.stat().st_size,**meta,"decoded_frames":len(frames)})
        decoded[rel]=(frames,ids)
    except Exception as exc:
        failures.append({"video_path":rel,"error":str(exc)})
video_df=pd.DataFrame(records)
print("sample success",len(records),"failures",failures)
video_df


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(13,4))
for ax,(_,row) in zip(axes,video_df.iterrows()):
    frames,indices=decoded[row["video_path"]]
    ax.imshow(frames[0]); ax.set_title(f"{row['video_path'].split('/')[-1]}\nframe {indices[0]}"); ax.axis("off")
plt.tight_layout()


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(13,3.5))
for column,ax in zip(["fps","duration_seconds","frame_count"],axes):
    ax.hist(video_df[column].dropna(),bins=min(5,max(1,len(video_df))))
    ax.set_title(f"Sample {column}")
plt.tight_layout()
print("resolution counts",video_df.groupby(["width","height"]).size().to_dict())
print("corrupted/unreadable in selected sample",len(failures))
